# TopoConscious — Validation & ROC Curves
Compares TopoConscious AUC against static functional connectivity baseline.

In [ ]:
import numpy as np
from topoconscious.validation import ValidationRunner


## Synthetic propofol dataset

In practice, load real BOLD time series here. For demonstration we generate synthetic data where the 'conscious' group has injected correlated activity.

In [ ]:
rng = np.random.default_rng(42)
N_SUBJECTS = 20
N_VOLS, N_REGIONS = 120, 30

def make_subject(conscious: bool, seed: int):
    rng2 = np.random.default_rng(seed)
    ts = rng2.standard_normal((N_VOLS, N_REGIONS))
    if conscious:
        # More correlated loops => higher H1 persistence
        for i in range(10):
            ts[:, i] += 0.6 * ts[:, 0]
    return ts

ts_list = [make_subject(i < N_SUBJECTS//2, seed=i) for i in range(N_SUBJECTS)]
labels = np.array([1]*(N_SUBJECTS//2) + [0]*(N_SUBJECTS//2))
print(f'Dataset: {N_SUBJECTS} subjects, labels: {dict(zip(*np.unique(labels, return_counts=True)))}')

In [ ]:
runner = ValidationRunner(output_dir='results/validation')
results = runner.evaluate_dataset(ts_list, labels, dataset_name='propofol_synthetic')
print(f"\nAUC TopoConscious : {results['auc_topo']:.3f}")
print(f"AUC Static FC     : {results['auc_fc']:.3f}")

In [ ]:
roc_path = runner.plot_roc_curves({'propofol_synthetic': results})
from IPython.display import Image
Image(roc_path)

## Batch evaluation on multiple datasets

In [ ]:
# To run on real BIDS datasets, replace with actual time-series loading:
# from bids import BIDSLayout
# layout = BIDSLayout('data/propofol_study')
# ...

all_results = runner.evaluate_all({
    'propofol': (ts_list[:10], labels[:10]),
    'sleep':    (ts_list[10:], labels[10:]),
})
runner.plot_roc_curves(all_results)